# A — Baselines and independent criticism (M1 / M2 / M3)

**Research question (part of RQ1):** Does independent criticism (M3) outperform self-reflection (M2) and direct generation (M1)?

**Owner:** Contributor A. **Status: no empirical results yet.** This notebook (1) runs an offline mock demo that checks the M1–M3 wiring, and (2) analyses the real results only after E's runner and evaluation have produced them. It never makes up numbers. If the result files are missing, the analysis cells stop with a message.

Inputs expected after integration (formats to be confirmed with E):
- `results/records/*.jsonl`: one `ExperimentRecord` per (example, method), written by E's shared serializer.
- `results/evaluation/per_example.csv`: E's official per-example metrics, with at least `example_id, method_id, status, em, f1`. Draft-level `draft_em` is needed for the transition analysis, and claim-factuality columns are optional.

A computes descriptive comparisons. Paired significance tests, CIs and the official numbers come from E.

In [ ]:
import sys, json, dataclasses
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'contracts').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.agents import answer as A, baselines as B, independent_critic as C
print('repo root:', ROOT)
print('prompt versions:', json.dumps(A.PROMPT_VERSIONS, indent=1))

## 1. Offline mock demo (wiring check only; these outputs are scripted, not results)

In [ ]:
from tests.agents.mocks_A import BRIDGE_EXAMPLE, MockLLMClient, MockRevisionPort, j

cfg = A.AgentConfig(model='mock/model')
llm = MockLLMClient({
    A.STAGE_DRAFT: j({'short_answer': 'Stockholm', 'explanation': 'Silver Harbor was directed by Mara Lind. Lind was born in Stockholm.'}),
    A.STAGE_SELF_CHECK: j({'issues': [], 'overall_comment': 'No issues found.'}),
    A.STAGE_SELF_REVISE: j({'short_answer': 'Stockholm', 'explanation': 'Silver Harbor was directed by Mara Lind. Lind was born in Stockholm.'}),
    A.STAGE_CRITIC: j({'issues': [{'rationale': 'Context says Lind was born in Gothenburg.', 'suggested_correction': 'Gothenburg'}], 'overall_comment': 'Wrong birthplace.'}),
})
agent = A.AnswerAgent(cfg)
draft = agent.generate_draft(BRIDGE_EXAMPLE, llm)          # shared draft, reused below
m1 = B.run_m1(BRIDGE_EXAMPLE, draft)
m2 = B.run_m2(BRIDGE_EXAMPLE, draft, llm, cfg)
m3 = B.run_m3(BRIDGE_EXAMPLE, draft, llm, cfg, MockRevisionPort())   # D's revision is mocked here
for out in (m1, m2, m3):
    print(out.method_id, '->', out.final_answer, '| feedback items:', len(out.feedback.items) if out.feedback else None)
print('LLM stages called:', llm.stages())

## 2. Load real experiment records (after E's run)

In [ ]:
REC_DIR = ROOT / 'results' / 'records'
EVAL_CSV = ROOT / 'results' / 'evaluation' / 'per_example.csv'
records = []
if REC_DIR.exists():
    for f in sorted(REC_DIR.glob('*.jsonl')):
        with f.open(encoding='utf-8') as fh:
            records += [json.loads(line) for line in fh if line.strip()]
records = [r for r in records if r.get('method_id') in {'M1', 'M2', 'M3'}]
HAVE_RESULTS = bool(records) and EVAL_CSV.exists()
print(f'{len(records)} M1-M3 records loaded; evaluation file present: {EVAL_CSV.exists()}')
if not HAVE_RESULTS:
    print('No real results yet: the analysis cells below will be skipped. Do not report any numbers.')

In [ ]:
if HAVE_RESULTS:
    import pandas as pd
    ev = pd.read_csv(EVAL_CSV)
    ev = ev[ev.method_id.isin(['M1', 'M2', 'M3'])]
    # Failures stay in the denominator (contract section 2): count status per method first.
    display(ev.groupby(['method_id', 'status']).size().unstack(fill_value=0))
    # Paired subset: examples that have an evaluated row for all three methods.
    piv = ev.pivot_table(index='example_id', columns='method_id', values=['em', 'f1'])
    paired = piv.dropna()
    print('paired examples:', len(paired), 'of', ev.example_id.nunique())
    display(paired.mean().unstack(0).round(4))
    if 'question_type' in ev.columns:
        display(ev.groupby(['question_type', 'method_id'])[['em', 'f1']].mean().round(4))

## 3. Revision transitions (draft → final)

wrong→correct counts as a successful correction, and correct→wrong as an error the revision introduced. This needs `draft_em` from E, which is EM of the shared draft and is identical across methods by construction.

In [ ]:
if HAVE_RESULTS and 'draft_em' in ev.columns:
    t = ev[ev.method_id.isin(['M2', 'M3'])].copy()
    t['transition'] = t.draft_em.map({1: 'correct', 0: 'wrong'}) + '→' + t.em.map({1: 'correct', 0: 'wrong'})
    display(t.groupby(['method_id', 'transition']).size().unstack(fill_value=0))
elif HAVE_RESULTS:
    print('draft_em not provided by E yet; ask E to add it to per_example.csv.')

## 4. Inspect erroneous critic feedback (M3 correct→wrong, and M3 missed errors)

In [ ]:
if HAVE_RESULTS and 'draft_em' in ev.columns:
    by_key = {(r['example_id'], r['method_id']): r for r in records}
    m3 = ev[ev.method_id == 'M3']
    harmful = m3[(m3.draft_em == 1) & (m3.em == 0)].example_id.tolist()
    missed = m3[(m3.draft_em == 0) & (m3.em == 0)].example_id.tolist()
    print(f'harmful critiques: {len(harmful)}; uncorrected wrong drafts: {len(missed)}')
    for ex_id in harmful[:10]:
        r = by_key[(ex_id, 'M3')]
        print('\n==', ex_id)
        print('draft:', r['draft']['short_answer'], '| final:', (r.get('final') or {}).get('final_answer'))
        for it in (r.get('feedback') or {}).get('items', []):
            print('  -', it['rationale'], '| fix:', it.get('suggested_correction'))

## 5. Notes for the report
- State sample size, split, backbone, and prompt versions (`A.PROMPT_VERSIONS`) next to every number.
- M2 = 2 calls (self-check + self-revise), M3 = 2 calls (critic + D revision), M1 = 0 extra calls. E reports cost/latency.
- Limitation: M2 uses A's own revision prompt, while M3 uses D's shared revision prompt. M2 vs M3 therefore changes both who critiques and how the revision is done.
- Report null or negative findings as valid outcomes.